# 00 — Download Raw PubChem Data

Downloads cytotoxicity assay data from PubChem:
- AID 1345082 → 3T3 (mouse fibroblast)
- AID 1345083 → HEK293 (human embryonic kidney)

Source: NCATS CellTiter-Glo luciferase assay, 48h incubation, ~90k compounds.

In [1]:
import os
import urllib.request
import zipfile
import pandas as pd

os.makedirs('../data/raw', exist_ok=True)

In [2]:

# PubChem FTP bulk download — real structure:
#   Bucket zip: Data/{start}_{end}.zip  (one zip per 1000 AIDs)
#   Inside zip: {start}_{end}/{aid}.csv.gz
#
# AID 1345082 and 1345083 are both in bucket 1345001_1346000.zip

import gzip, io

ASSAYS = {
    '1345082': '3T3',
    '1345083': 'HEK293',
}

FTP_BASE = 'https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/CSV/Data/'

def aid_to_bucket(aid_str):
    aid = int(aid_str)
    start = (aid // 1000) * 1000 + 1
    end   = start + 999
    return f'{start}_{end}'

# Group AIDs by bucket so we download each bucket zip only once
from collections import defaultdict
buckets = defaultdict(list)
for aid, name in ASSAYS.items():
    buckets[aid_to_bucket(aid)].append((aid, name))

for bucket, aids in buckets.items():
    bucket_url  = f'{FTP_BASE}{bucket}.zip'
    bucket_file = f'../data/raw/bucket_{bucket}.zip'

    # Skip if all CSVs for this bucket already exist
    if all(os.path.exists(f'../data/raw/{name}_raw.csv') for _, name in aids):
        print(f'Bucket {bucket}: all files already downloaded, skipping.')
        continue

    print(f'Downloading bucket {bucket} (~20 MB) from {bucket_url} ...')
    urllib.request.urlretrieve(bucket_url, bucket_file)
    print(f'  Done. Extracting target AIDs ...')

    with zipfile.ZipFile(bucket_file, 'r') as z:
        for aid, name in aids:
            csv_path = f'../data/raw/{name}_raw.csv'
            if os.path.exists(csv_path):
                print(f'  {name}: already exists, skipping.')
                continue

            member = f'{bucket}/{aid}.csv.gz'
            print(f'  Extracting {member} ...')
            gz_data = z.read(member)
            csv_data = gzip.decompress(gz_data)
            with open(csv_path, 'wb') as f:
                f.write(csv_data)
            print(f'  {name}: saved to {csv_path}')

    os.remove(bucket_file)
    print(f'Bucket zip removed.')


Bucket 1345001_1346000: all files already downloaded, skipping.


In [3]:
# Quick sanity check
for name in ASSAYS.values():
    df = pd.read_csv(f'../data/raw/{name}_raw.csv', low_memory=False)
    print(f'{name}: {len(df):,} rows, columns: {list(df.columns[:8])}')

3T3: 93,781 rows, columns: ['PUBCHEM_RESULT_TAG', 'PUBCHEM_SID', 'PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_URL', 'PUBCHEM_ASSAYDATA_COMMENT']
HEK293: 93,781 rows, columns: ['PUBCHEM_RESULT_TAG', 'PUBCHEM_SID', 'PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_URL', 'PUBCHEM_ASSAYDATA_COMMENT']
